In [1]:
#SHAP with an Iterator
import itertools

# Define the base scores of the words
words = {'I': 0.2, 'love': 0.6, 'playing': 0.5, 'chess': 0.4, 'with': 0.1, 'my': 0.3, 'friends': 0.4}

# Define the bonus scores for certain combinations of words
bonus = {('I', 'love'): 0.3, ('love', 'playing'): 0.25, ('playing', 'chess'): 0.45, ('with', 'my', 'friends'): 0.35}

In [2]:
# Function to calculate the total score of a coalition
def total_score(coalition):
    score = sum(words[word] for word in coalition)
    for b in bonus.keys():
        if all(word in coalition for word in b):
            score += bonus[b]
    return score

In [3]:
# Function to calculate the Shapley value of a word
def shapley_value(word):
    N = len(words)
    permutations = list(itertools.permutations(words))
    marginal_contributions = []
    counter = 0 # Counter initialization
    for permutation in permutations:
      index = permutation.index(word)
      coalition_without_word = permutation[:index]
      coalition_with_word = permutation[:index+1]
      marginal_contribution = total_score(coalition_with_word) - total_score(coalition_without_word)
      marginal_contributions.append(marginal_contribution)
      counter += 1 # Increment counter
    print(f"Processed {counter} permutations") # Print counter
    return sum(marginal_contributions)

In [4]:
# Calculate the Shapley value of each word
for word in words:
  print(f"The Shapley value of '{word}' is {shapley_value(word)}")

Processed 5040 permutations
The Shapley value of 'I' is 1764.0
Processed 5040 permutations
The Shapley value of 'love' is 4410.0
Processed 5040 permutations
The Shapley value of 'playing' is 4284.0
Processed 5040 permutations
The Shapley value of 'chess' is 3150.0
Processed 5040 permutations
The Shapley value of 'with' is 1092.0000000000002
Processed 5040 permutations
The Shapley value of 'my' is 2099.9999999999995
Processed 5040 permutations
The Shapley value of 'friends' is 2604.0


In [10]:
#Hugging Face Transformers
!pip install transformers
#Transformer building blocks
!pip install xformers
#SHAP
!pip install shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.7 MB/s eta 0:00:00


In [11]:
#@title Enter your sentence here:
sentence = 'SHAP is a useful explainer' #@param {type:"string"}

In [12]:
import transformers

# load a transformers pipeline model
model = transformers.pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

# analyze the sentiment of the input sentence
result = model(sentence)[0]
print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

{'label': 'POSITIVE', 'score': 0.9869391918182373}


In [15]:
import shap

# explain the model on the input sentence
explainer = shap.Explainer(model)
shap_values = explainer([sentence])

# visualize the first prediction's explanation for the predicted class
predicted_class = result['label']
shap.plots.text(shap_values[0, :, predicted_class])

In [16]:
# get the sentiment score
sentiment_score = model(sentence)[0]['score']

# print the SHAP values for each word
words = sentence.split(' ')
for word, shap_value in zip(words, shap_values.values[0, :, 0]):
  print(f"Word: {word}, SHAP value: {shap_value}")

Word: SHAP, SHAP value: 0.0
Word: is, SHAP value: 0.1244606003165245
Word: a, SHAP value: 0.17050934582948685
Word: useful, SHAP value: -0.11854981258511543
Word: explainer, SHAP value: -0.09484544023871422
